# 04.02 — Case Study: Summarization Berita — Groq vs TinyLlama Q4

**Tujuan**: bandingkan 2 pendekatan untuk task generative (summarization artikel berita Bahasa):
- **LLM via API** (Groq llama-3.1-8b) — biaya per call, latency rendah, kualitas tinggi
- **SLM Quantized lokal** (TinyLlama-1.1B Q4) — gratis, offline-able, kualitas lebih rendah

**Prasyarat**: notebook 02.03 lulus (TinyLlama GGUF sudah di-download).

**Sample size**: 5 artikel dari Liputan6 (cukup untuk insight; kalau lebih, terlalu lama TinyLlama-nya).

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Siapkan 5 artikel

Coba load Liputan6 dari HuggingFace; fallback ke artikel hardcoded kalau dataset tidak available.

In [ ]:
FALLBACK_ARTICLES = [
    {
        "id": 1,
        "text": (
            "Jakarta - Bank Indonesia (BI) memutuskan untuk menahan suku bunga acuan BI Rate "
            "di level 5,75 persen pada Rapat Dewan Gubernur bulan ini. Keputusan ini diambil "
            "sejalan dengan upaya menjaga stabilitas nilai tukar rupiah serta meredam tekanan "
            "inflasi di tengah ketidakpastian ekonomi global. Gubernur BI menyatakan bahwa "
            "kebijakan moneter ke depan akan tetap berhati-hati dan data-dependent."
        ),
    },
    {
        "id": 2,
        "text": (
            "Bandung - Tim peneliti Institut Teknologi Bandung (ITB) berhasil mengembangkan "
            "baterai berbahan dasar air laut yang diklaim ramah lingkungan dan lebih murah "
            "dibanding lithium-ion. Inovasi ini telah melewati pengujian skala laboratorium "
            "dan kini memasuki tahap uji prototipe industri. Pemerintah memberikan dukungan "
            "melalui dana riset Kemendikbudristek senilai 3 miliar rupiah."
        ),
    },
    {
        "id": 3,
        "text": (
            "Surabaya - Hujan deras yang mengguyur sejumlah wilayah Surabaya sejak Senin malam "
            "menyebabkan banjir di 12 titik. Ketinggian air bervariasi antara 30 sentimeter "
            "hingga 1 meter. Pemerintah Kota Surabaya telah menerjunkan tim BPBD untuk evakuasi "
            "warga dan membersihkan saluran air. Sekolah-sekolah di daerah terdampak diliburkan "
            "sementara hingga kondisi pulih."
        ),
    },
    {
        "id": 4,
        "text": (
            "Yogyakarta - Festival Film Indonesia 2026 akan dibuka pada bulan November di "
            "Yogyakarta dengan tema 'Kembali ke Akar Budaya'. Lebih dari 200 film dari sineas "
            "Indonesia dan negara ASEAN akan ditayangkan. Festival ini juga menyediakan ruang "
            "diskusi panel dengan sutradara senior serta workshop untuk pembuat film pemula."
        ),
    },
    {
        "id": 5,
        "text": (
            "Jakarta - Aplikasi e-commerce lokal Tokoraya melaporkan pertumbuhan transaksi 45 "
            "persen pada kuartal pertama 2026 dibanding periode yang sama tahun lalu. Pertumbuhan "
            "didorong oleh ekspansi ke kota tier-2 dan tier-3 serta peningkatan kategori produk "
            "lokal. CEO Tokoraya menyatakan optimisme target tembus 1 juta transaksi per hari pada akhir tahun."
        ),
    },
]

try:
    from datasets import load_dataset
    ds = load_dataset("SEACrowd/liputan6_canonical", split="validation[:5]", trust_remote_code=True)
    text_key = "text" if "text" in ds.column_names else ("document" if "document" in ds.column_names else None)
    if text_key is None:
        raise RuntimeError("Schema dataset tidak sesuai")
    articles = [{"id": i, "text": ds[i][text_key]} for i in range(len(ds))]
    print(f"Loaded {len(articles)} artikel dari Liputan6 dataset.")
except Exception as e:
    print(f"[INFO] Liputan6 load gagal ({e}); pakai 5 artikel fallback hardcoded.")
    articles = FALLBACK_ARTICLES

# truncate kalau terlalu panjang
for a in articles:
    if len(a["text"]) > 1500:
        a["text"] = a["text"][:1500]
print(f"\nTotal artikel: {len(articles)}")

## 2. Setup 2 backend

In [ ]:
import time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

groq = groq_client()

gguf_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    local_dir=str(repo_root / "models"),
)
tiny = Llama(model_path=gguf_path, n_ctx=2048, n_threads=4, verbose=False)
print("Both backends ready.")

## 3. Summarize tiap artikel di 2 backend

In [ ]:
SUMMARY_PROMPT_TEMPLATE = (
    "Ringkas artikel berita berikut dalam 2 kalimat singkat, dalam Bahasa Indonesia. "
    "Fokus ke fakta utama saja, tidak perlu opini.\n\n"
    "Artikel:\n{text}\n\n"
    "Ringkasan (2 kalimat):"
)

def summarize_groq(text: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    r = groq.chat.completions.create(
        model=GROQ_DEFAULT_MODEL,
        messages=[{"role": "user", "content": SUMMARY_PROMPT_TEMPLATE.format(text=text)}],
        max_tokens=120, temperature=0,
    )
    ms = (time.perf_counter() - t0) * 1000
    return r.choices[0].message.content.strip(), ms

def summarize_tiny(text: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    r = tiny.create_chat_completion(
        messages=[{"role": "user", "content": SUMMARY_PROMPT_TEMPLATE.format(text=text)}],
        max_tokens=120, temperature=0,
    )
    ms = (time.perf_counter() - t0) * 1000
    return r["choices"][0]["message"]["content"].strip(), ms

from tqdm.auto import tqdm
results = []
for a in tqdm(articles, desc="summarize"):
    groq_summary, groq_ms = summarize_groq(a["text"])
    tiny_summary, tiny_ms = summarize_tiny(a["text"])
    results.append({
        "id": a["id"], "original": a["text"],
        "groq_summary": groq_summary, "groq_ms": round(groq_ms),
        "tiny_summary": tiny_summary, "tiny_ms": round(tiny_ms),
    })
print(f"\nDone — {len(results)} artikel.")

## 4. Display side-by-side

In [ ]:
for r in results:
    print("=" * 80)
    print(f"ARTIKEL #{r['id']} (original {len(r['original'])} chars)")
    print(r["original"][:200] + ("..." if len(r["original"]) > 200 else ""))
    print(f"\n[GROQ — {r['groq_ms']} ms]")
    print(r["groq_summary"])
    print(f"\n[TINYLLAMA Q4 — {r['tiny_ms']} ms]")
    print(r["tiny_summary"])
print("=" * 80)

## 5. Agregat metrik

Tanpa ROUGE (butuh reference summary). Kita pakai metrik proxy: panjang output (tokens estimasi), latency rata-rata.

In [ ]:
import pandas as pd
import numpy as np

summary_df = pd.DataFrame([
    {
        "backend": "groq",
        "latency_ms_mean": np.mean([r["groq_ms"] for r in results]),
        "output_chars_mean": np.mean([len(r["groq_summary"]) for r in results]),
    },
    {
        "backend": "tinyllama_q4",
        "latency_ms_mean": np.mean([r["tiny_ms"] for r in results]),
        "output_chars_mean": np.mean([len(r["tiny_summary"]) for r in results]),
    },
]).round(1)
summary_df["speedup_groq_vs_tiny"] = [
    1.0,
    round(summary_df.iloc[1]["latency_ms_mean"] / summary_df.iloc[0]["latency_ms_mean"], 1),
]
summary_df

## Refleksi & insight

1. **Untuk task generative kualitas natural Bahasa**, Groq jauh menang. TinyLlama Q4 yang kecil sering ngalor-ngidul, switch ke English, atau ulangi artikel.
2. **Tapi kalau privacy hard requirement** (artikel internal perusahaan), TinyLlama lokal jadi pilihan. Quality cukup untuk "extractive summary" sederhana.
3. **Latency Groq jauh lebih cepat** — sering 5-30x. Karena LPU + jaringan optimasi.
4. **Biaya summarization** lebih tinggi dari klasifikasi (output panjang). Tapi masih kecil per call ($0.0001-0.001).
5. **Untuk production summarization Bahasa**, opsi terbaik biasanya: LLM API dengan fine-tuned prompt + caching.

## Latihan mandiri

1. Coba pakai prompt yang lebih spesifik: "Ringkas dalam bullet list 3 poin: SIAPA, APA, KAPAN". Apakah TinyLlama jadi lebih reliable dengan struktur?
2. Tambah backend SmolLM2-135M fp32. Kualitas-nya seberapa parah dibanding TinyLlama Q4 untuk Bahasa?
3. Pakai ROUGE proper (butuh `evaluate` library + reference summary dari Liputan6). Hitung ROUGE-L Groq vs TinyLlama vs reference.

## Lanjut

Case study terakhir: mini RAG — LLM dengan retrieved context: [03_qa_dengan_konteks_mini_rag.ipynb](03_qa_dengan_konteks_mini_rag.ipynb)